In [0]:
pip install Faker

In [0]:
# =============================================================
# NOTEBOOK:  02_bronze_batch_ingestion_products
# PURPOSE:   Auto Loader pipeline for ERP products data
# SOURCE:    ERP System → CSV files
# TARGET:    Bronze Delta Table (bronze/erp/products/)
# AUTHOR:    Your Name
# DATE:      Today's date
# PROJECT:   RetailMax Lakehouse
# =============================================================

import json
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
from faker import Faker
import random
import uuid



# ── Load Config ───────────────────────────────────────────────
config_path = "/Workspace/Repos/retailmax-lakehouse/retailmax-lakehouse/configs/dev/config.json"

with open(config_path, 'r') as f:
    config = json.load(f)

# ── Storage Paths ─────────────────────────────────────────────
storage_account = config['storage']['account_name']
bronze_path     = config['storage']['bronze_path']
scope_name      = config['secret_scope']

# ── Retrieve Secrets ──────────────────────────────────────────
client_id     = dbutils.secrets.get(scope=scope_name, key="sp-client-id")
client_secret = dbutils.secrets.get(scope=scope_name, key="sp-client-secret")
tenant_id     = dbutils.secrets.get(scope=scope_name, key="sp-tenant-id")

# ── Configure Spark for ADLS ──────────────────────────────────
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
    client_id
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
    client_secret
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

print("=" * 60)
print("RetailMax - Bronze Batch Ingestion: Products")
print("=" * 60)
print(f"✅ Config loaded")
print(f"✅ Secrets retrieved")
print(f"✅ ADLS configured")
print(f"   Bronze Path: {bronze_path}")
print("=" * 60)


In [0]:
source_path = bronze_path + 'raw_ingest/erp/products/'
target_path = bronze_path + "tables/erp/products/"
checkpoint_path = bronze_path + "_checkpoints/erp/products/"
schema_path = bronze_path + "_schema/erp/products/"
bad_records_path = bronze_path + "_quarantine/erp/products/"
test_path = bronze_path + "tables/test/schema_evolution_products/"


In [0]:
fake = Faker()

CATEGORIES = {
    "Apparel":       ["Tops", "Bottoms", "Footwear"],
    "Electronics":   ["Phones", "Laptops", "Audio"],
    "Home & Garden": ["Furniture", "Kitchen", "Decor"],
}

def generate_products(start_id, n=500):
    products = []
    for i in range(start_id, start_id + n):
        main_cat = random.choice(list(CATEGORIES.keys()))
        sub_cat  = random.choice(CATEGORIES[main_cat])
        products.append({
            "product_id":   i,
            "product_name": fake.catch_phrase(),
            "sku":          f"SKU-{i:06d}",
            "category": {
                "main": main_cat,
                "sub":  sub_cat,
                "tags": random.sample(["new","sale","clearance","premium"], k=2)
            },
            "pricing": {
                "cost_price":   round(random.uniform(5.0, 100.0), 2),
                "retail_price": round(random.uniform(10.0, 300.0), 2),
                "currency":     random.choice(["USD", "EUR", "GBP"])
            },
            "supplier":       fake.company(),
            "stock_quantity": random.randint(0, 5000),
            "is_active":      random.choice([True, False])
        })
    return products

batch1 = generate_products(start_id=1,   n=500)
batch2 = generate_products(start_id=501, n=500)

# Save to ADLS as JSON Lines
def save_json(data, folder):
    json_lines = "\n".join([json.dumps(r) for r in data])
    path = source_path + folder + "/products.json"
    dbutils.fs.put(path, json_lines, overwrite=True)
    print(f"✅ Saved {len(data)} records → {path}")

save_json(batch1, "products_batch1")
save_json(batch2, "products_batch2")

In [0]:
product_schema = """
    product_id INT,
    product_name STRING,
    sku STRING,
    category STRUCT<
        main: STRING,
        sub: STRING,
        tags: ARRAY<STRING>
    >,
    pricing STRUCT<
        cost_price: DOUBLE,
        retail_price: DOUBLE,
        currency: STRING
    >,
    supplier STRING,
    stock_quantity INT,
    is_active BOOLEAN
"""

In [0]:
# Reading with Auto Loader
df = spark.readStream\
    .schema(product_schema)\
    .format("cloudFiles")\
    .option("cloudFiles.format", "json")\
    .option("cloudFiles.schemaLocation", schema_path)\
    .load(source_path)

# Metadata columns
df = (df
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_load_timestamp", F.current_timestamp())
    .withColumn("_load_date", F.current_date())
    .withColumn("_batch_id", F.lit(uuid.uuid4().hex))
)

# Writing stream
df.writeStream\
    .format("delta")\
    .option("checkpointLocation", checkpoint_path)\
    .trigger(availableNow=True)\
    .partitionBy("_load_date")\
    .start(target_path)\
    .awaitTermination()

In [0]:
df = spark.read.format("delta").load(target_path)
print(df.count())
df.show(5)
df.printSchema()

In [0]:
df = spark.read.format("delta").load(target_path)
df.select("category.main", "category.sub", "category.tags").show(5)

In [0]:
df_existing = spark.read.format("delta")\
    .load(bronze_path + "tables/erp/products/")
df_existing.printSchema()

In [0]:
new_data = spark.createDataFrame([
    (9999, "Test Product", "SKU-9999", "extra_value")
], ["product_id", "product_name", "sku", "new_mystery_column"])

In [0]:
new_data.write\
    .mode("append")\
    .format("delta")\
    .save(bronze_path + "tables/erp/products/")

In [0]:
# Read existing
df_existing = spark.read.format("delta")\
    .load(bronze_path + "tables/erp/products/")\
    .limit(2)

# Add new column
df_with_new_col = df_existing.withColumn("new_mystery_column", F.lit("test_value"))

#Try to append
# df_existing.write\
#     .mode("append")\
#     .option("mergeSchema", "true")\
#     .format("delta")\
#     .save(test_path)

# Try to append
df_with_new_col.write\
    .mode("append")\
    .option("mergeSchema", "true")\
    .format("delta")\
    .save(test_path)

In [0]:
df_new = spark.read.format("delta")\
    .load(test_path)
df_new.show()
df_new.printSchema()